In [1]:
%pip install transformers torch torchvision torchaudio numpy pandas tqdm matplotlib huggingface_hub datasets evaluate scikit-learn accelerate


[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import sys
import os

# Path to the folder you want to add
subfolder_path = os.path.join(os.getcwd(), "roberta_labeller")

# Add it to sys.path
if subfolder_path not in sys.path:
    sys.path.append(subfolder_path)

In [3]:
import json
import os
import torch
import sys
import numpy as np
import pandas as pd
from transformers import (
    AutoModelForTokenClassification,
)

from roberta_labeller.dataset import prepare_datasets
from roberta_labeller.train import train_binary, evaluate_binary
from roberta_labeller.finetuning import save_results, sampling_tuning
from roberta_labeller.seeding import enforce_reproducibility

results_dir = "roberta_labeller_results"
if not os.path.exists(results_dir):
    os.makedirs(results_dir)

model_name = "FacebookAI/xlm-roberta-base"
oversample_ratio = 0.0
undersample_ratio = 0.0

/Users/xk84vl/Documents/Repos/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/xk84vl/Documents/Repos/.venv/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
enforce_reproducibility(42)

# ==== PREPARE DATASETS ====
train_set, val_set, test_sets, tokenizer = prepare_datasets(model_name, oversample_ratio, undersample_ratio, None)

# ==== MODEL ====
model = AutoModelForTokenClassification.from_pretrained(
    model_name, 
)

# ==== TRAIN MODEL ====
model, tokenizer, epoch_history = train_binary(model, train_set, val_set, tokenizer, 1, results_dir)

# ==== EVALUATE MODEL ====
eval_results = evaluate_binary(model, tokenizer, test_sets["ko"])

# ==== SAVE RESULTS & MODEL ====
save_results(results_dir, epoch_history, eval_results)

[2025-10-28 17:36:20,277] - [INFO] - Found 3 mislabelled instances


Creating CSV from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 856.68ba/s]
Creating CSV from Arrow format: 0ba [00:00, ?ba/s]

[2025-10-28 17:36:20,341] - [INFO] - Original Train dataset class split: Counter({1: 5375, 0: 123}), with a total of 5498 samples
[2025-10-28 17:36:20,455] - [INFO] - Sampled Train dataset class split: Counter({1: 5375, 0: 123}), with a total of 5498 samples
[2025-10-28 17:36:20,459] - [INFO] - Validation dataset class split: Counter({1: 597, 0: 14}), with a total of 611 samples
[2025-10-28 17:36:20,466] - [INFO] - Test dataset class split: Counter({1: 991, 0: 105}), with a total of 1096 samples



Map: 100%|██████████| 344/344 [00:00<00:00, 8168.30 examples/s]
Some weights of XLMRobertaForTokenClassification were not initialized from the model checkpoint at FacebookAI/xlm-roberta-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


[2025-10-28 17:36:22,331] - [INFO] - Starting training of model (answerability classification).


/Users/xk84vl/Documents/Repos/.venv/lib/python3.9/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss,Validation Loss


RuntimeError: MPS backend out of memory (MPS allocated: 4.70 GiB, other allocations: 22.95 GiB, max allowed: 27.20 GiB). Tried to allocate 256 bytes on shared pool. Use PYTORCH_MPS_HIGH_WATERMARK_RATIO=0.0 to disable upper limit for memory allocations (may cause system failure).